# S1 - Arquitectura Big Data: Lambda y Kappa, batch vs. streaming

**Actividad:** instalar y verificar `uso-pyspark`, y con el entorno funcionando, analizar `lambda26` y decidir entre arquitectura Lambda o Kappa, justificando batch vs. streaming para un caso de negocio (sílabo).

**Propósito de la actividad:** confirmar que el entorno del laboratorio funciona de punta a punta (Docker, Jupyter, Spark), y aplicar la regla de decisión batch/streaming a los casos de uso reales de `lambda26` (`orden-eventos`, `pago-eventos`) para seleccionar y justificar una arquitectura Big Data, proponiendo las tecnologías coherentes con esa elección.

Guía completa: `docs/sesiones/S01_Arquitectura_Big_Data_Lambda_Kappa.md`, sección 3 (pasos 3.1 a 3.7).

## 3.2 Verificar Spark con un notebook mínimo

**Producto del paso:** notebook con una SparkSession activa, un dataset cargado y visible en Spark UI.

### 1. Crear la SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s1-verificacion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark

### 2. Cargar un dataset como DataFrame

`biblia_ntv_.csv` ya está en `s01-arquitectura/data/` (montado como `/opt/s01-arquitectura/data/` en el contenedor).

In [ ]:
df = spark.read.csv("/opt/s01-arquitectura/data/biblia_ntv_.csv", header=True, inferSchema=True)
df

### 3. Ver la estructura y las primeras filas

In [ ]:
df.printSchema()
df.show(5, truncate=False)

### 4. (Opcional) una transformación simple

In [ ]:
df.select("libro", "capitulo", "verso").show(5)

**Verificación:** confirma en `http://localhost:4042` (Spark UI) que aparece el job que se acaba de ejecutar (la lectura del CSV y el `show()`).

## 3.3 Reconocer el ecosistema de `lambda26`

**Producto del paso:** mapa del flujo tecnológico del laboratorio.

Flujo base del ecosistema:

```
Usuarios -> Kafka -> Spark Processing -> Data Lake / Base de datos -> Dashboard / Aplicaciones
```

**1. ¿Dónde nacen los eventos?**
En los productores: `uso-rapido` (`ec-orden-py`) y `uso-ms-sb` (`ec-orden-ms`, `ec-pago-ms`), que generan `orden-eventos` y `pago-eventos`.

**2. ¿Qué componente ingesta los eventos?**
`kafka` — la columna vertebral de ingesta del laboratorio.

**3. ¿Qué componente procesa los datos a escala?**
`uso-pyspark` (Spark/PySpark), tanto en su parte batch como en su parte streaming.

**4. ¿Dónde se consumen los resultados?**
En `obs` (Prometheus + Grafana) para observabilidad/BI, y en las bases Postgres propias de cada microservicio (`orden_db`, `pago_db`) para el histórico.

## 3.4 Analizar el caso guiado y clasificar batch/streaming

**Producto del paso:** clasificación justificada del caso.

Caso: `uso-rapido` (`ec-orden-py` publica y consume `orden-eventos` por Kafka) y `uso-ms-sb` (`ec-orden-ms` publica `orden-eventos`; `ec-pago-ms` consume `orden-eventos` y publica `pago-eventos`; cada microservicio guarda su propio histórico en su base Postgres).

**¿Batch, streaming o ambos?** Ambos.

**Justificación:**
1. Cada microservicio guarda su propio histórico en Postgres — eso es un requisito batch/analítico sobre datos ya acumulados.
2. Los eventos se publican y consumen en tiempo real por Kafka (una orden creada dispara el procesamiento del pago) — eso es un requisito streaming genuino, no solo histórico.

## 3.5 Aplicar la regla de decisión

**Producto del paso:** arquitectura seleccionada y justificada.

Regla de decisión (2.5.1): si el caso necesita histórico + tiempo real → **Lambda**; si todo el caso son eventos en tiempo real → Kappa.

**Arquitectura seleccionada: Lambda.** El caso de 3.4 necesita histórico (Postgres por microservicio) y tiempo real (eventos por Kafka) a la vez, así que corresponde Lambda, no Kappa.

## 3.6 Proponer tecnologías y diagrama de flujo

**Producto del paso:** lista de tecnologías y diagrama de flujo simple.

**Tecnologías propuestas:**
- Kafka: ingesta de `orden-eventos` y `pago-eventos`.
- Spark Streaming: speed layer, procesamiento en vivo de los eventos.
- Spark Batch: batch layer, sobre el histórico acumulado en Postgres.
- Postgres / Data Lake: almacenamiento (histórico por microservicio + capa RAW).
- Grafana: BI en tiempo real (serving layer / visualización).

**Diagrama de flujo:**

```
orden-eventos / pago-eventos -> Kafka -> Spark Streaming + Batch -> Data Lake / Postgres -> Grafana
```

## 3.7 Completar la plantilla de propuesta

**Producto del paso:** ficha de propuesta arquitectónica completa.

**Tabla 5. Ficha de propuesta arquitectónica**

| Campo | Completa |
|---|---|
| **Caso analizado** | Plataforma de e-commerce interna de `lambda26`: cada compra genera un evento de orden (`ec-orden-ms`/`ec-orden-py`) y, tras el pago, un evento de pago (`ec-pago-ms`); cada microservicio guarda su propio histórico en Postgres. |
| **Tipo de procesamiento** | Ambos — batch (histórico en Postgres por microservicio) + streaming (eventos de orden/pago en tiempo real vía Kafka). |
| **Arquitectura seleccionada** | Lambda — el caso necesita histórico y tiempo real a la vez, según la regla de decisión de 2.5.1. |
| **Diagrama de arquitectura** | `orden-eventos / pago-eventos -> Kafka -> Spark Streaming + Batch -> Data Lake / Postgres -> Grafana` |
| **Tecnologías propuestas** | Kafka (ingesta) → Postgres por microservicio + Data Lake (almacenamiento) → Spark Batch + Streaming (procesamiento) → Grafana (visualización). |
| **Supuestos y riesgos** | Supuestos: los microservicios `ec-orden-ms`/`ec-pago-ms` generan eventos de forma continua; Kafka soporta el volumen del laboratorio. Riesgos: mantener consistencia entre la capa batch y la capa streaming (*drift risk* propio de Lambda, ver 2.5.1); mayor complejidad operativa por sostener tres capas (batch/speed/serving). |

**Evidencia de aprendizaje:**
- Entorno `lambda26` (`uso-pyspark`) funcionando y verificado, con un notebook que muestra Spark UI activo.
- Clasificación batch/streaming y arquitectura (Lambda) seleccionada, con justificación.
- Ficha de propuesta arquitectónica completa (tecnologías, diagrama de flujo, supuestos y riesgos).